In [ ]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from torchvision.datasets import MNIST
from torchvision.transforms import v2

In [ ]:
train_tmsf = v2.Compose([
    v2.Resize((128,128)),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(15),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale = True),
    v2.ColorJitter(brightness = 0.5, contrast = 0.5),
    v2.Normalize(mean = (0.5, 0.5, 0.5), std = (0.5, 0.5, 0.5))
    ])

val_tmsf = v2.Compose([
    v2.Resize((128,128)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale = True),
    v2.Normalize(mean = (0.5, 0.5, 0.5), std = (0.5, 0.5, 0.5))
])

In [ ]:
train_data = MNIST(root="./data", train = True, download = True, transform = train_tmsf)
test_data = MNIST(root = "./data", train = True, download = True, transform = val_tmsf)

In [ ]:
genrator = torch.Genrator().manual_seed(42)

train_set, val_set = random_split(
    train_data,
    [0.8, 0.2],
    genrator = genrator
)

In [ ]:
train_loader = DataLoader(train_set, shuffle = True, batch_size = 42)
val_loader = DataLoader(val_set, shuffle = False, batch_size = 42)
test_loader = DataLoader(test_data, shuffle = False, batch_size = 42)

In [ ]:
class Handwritten_Digit(nn.Module):
    def __init__(self, num_classes = 9):
        super().__init__()
        self.num_classes = num_classes


        self.model = nn.Sequential(
            nn.Con2d(3, 32, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            
            nn.Conv2d(64, 128, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            
            nn.Conv2d(128, 256, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)

      )

        self.fc_layer = nn.Sequential(
            nn.Linear(8*8*256, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, self.num_classes)
        )

    def forward(self, x):
        x = self.model(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layer(x)
        return x

In [ ]:
model = Handwritten_Digit()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())